<a href="https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/CIS_5450_Project_Difficulty_Topics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIS 5450 Project: Difficulty Topics
**Group Members:**
* **Shangyi Du**
* **Jingyi Gong**
* **Chenning Huang**

> This notebook documents how you implemented difficulty topics in your project. Use the link button in the top right when you select a cell to get a **hyperlink**.


## Topic 1: Entity Linking
[Hyperlink](https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/ufo_sightings.ipynb#scrollTo=mH3thvhSHusP&line=1&uniqifier=1)

### Why we used this concept
We applied entity linking because our research aims to examine whether contextual factors beyond direct observations, such as environmental conditions, geographic context, and cultural or socioeconomic characteristics, may influence how UFO shapes are reported. The NUFORC sightings dataset alone lacks these additional predictors, so we incorporated multiple external datasets at the city and country levels. Since these contextual features are not directly aligned to sighting coordinates, entity linking was necessary to accurately assign the nearest valid environmental and human-contextual attributes to each UFO report. This enriched dataset provides a more comprehensive feature space that enables us to explore a wider range of potential influences in subsequent modeling and analysis.

### How we implemented it
We applied two entity linking:

1. **UFO Sightings → City Features (Nearest-Valid City Matching)**  
   In the city features dataset, `population`, `avg_annual_temp(°C)`, `temp_seasonality`, `annual_precipitation(mm)`, `elevation(m)`, and `viirs_annual_ave` contain missing values for some cities. If we simply merged each UFO sighting with the single nearest city, some UFO records would inherit missing feature values even though a slightly more distant nearby city has complete data.

   To address this, we implemented a nearest-non-null matching procedure:

   - For each UFO sighting, we identified the k nearest cities using BallTree K-nearest-neighbor search in projected coordinates.

   - For each feature, we examined these candidate cities in order of distance (nearest → farther).

   - We selected the closest city that actually has a valid (non-null) value for that feature.

   - We also recorded the distance to the city that supplied the data, producing a meaningful measure of spatial accuracy.


2. **UFO Sightings → Country Education Data (Reverse Geocoding)**  
   Because education indicators are reported at the country level, we:

   - Cleaned and numeric-converted the latitude/longitude fields.

   - Reverse-geocoded coordinates into standardized ISO-aligned country labels.

   - Merged the dataset via a normalized "country" key.


### Results & Interpretation
Entity linking allowed us to successfully augment each UFO sighting with environmental, geographic, and human-contextual features while avoiding missing-value propagation. By assigning the closest valid city for each contextual variable, this approach preserves geographic proximity and ensures that we do not introduce incomplete or unrealistic data. It also standardizes national attributes through reverse geocoding, associating each sighting with consistent country-level cultural and socioeconomic indicators. As a result, we obtained a more complete and robust feature set—one that maintains spatial realism, resolves missing data issues, and provides a reliable foundation for downstream modeling and analysis.

## Topic 2: Imbalance data (SMOTE)
[Hyperlink](https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/ufo_sightings.ipynb#scrollTo=kkAI_oyt5zE8)
### Why we used this concept
Our target variable has 21 categories with a highly imbalanced distribution. A few shapes (especially "light" and "triangle") account for a large fraction of the sightings, while many other shapes (e.g., "cone", "cross", "teardrop") are rare.

When we trained models on the original data, we observed that:


*   The majority-class baseline already achieved 21% accuracy by always predicting "light".
*   More complex models tended to focus on the dominant classes and almost ignore the rare ones, leading to low macro-style F1 scores.

To address this, we implemented SMOTE (Synthetic Minority Oversampling TEchnique). This allows us to:

*   Give each shape class a sufficient number of training examples.
*   Encourage models to discover more patterns in minority classes.





### How we implemented it


### Results & Interpretation
SMOTE had a very clear effect on training metrics, but only a limited effect on test performance.

For tree-based models (e.g. Random Forest and AdaBoost), SMOTE dramatically increased training accuracy and weighted macro F1 (even above 0.9), showing that the models can easily fit the oversampled data.

On the test set, however, accuracy improved only slightly. XGBoost benefited the most from SMOTE. With SMOTE, XGBoost's test accuracy rose to about 16.4% and weighted F1 improved to 0.125, which is still below the 21% majority baseline but better than the original XGBoost performance.

For SVM, SMOTE improved training metrics on the subsampled training set, but test accuracy remained around 4 - 5% with very low weighted macro F1, indicating that the SVM still struggles in this high-dimensional, noisy, multi-class setting.

Overall, SMOTE successfully balanced the training distribution and made minority classes more visible during training. However, it did not fully solve the generalization problem.

This suggests that, for our UFO shape prediction task, the main limitation is the weak signal in the current features rather than the imbalance technique itself. SMOTE is helpful for exploration, but it cannot create class separability where there is very little information to begin with.


## Topic 3: Ensemble Models (Random Forests and XGBoost)
[Hyperlink](https://colab.research.google.com/github/jingyig16/ufo-sightings/blob/main/ufo_sightings.ipynb#scrollTo=pkSnLbK4hK4U)
### Why we used this concept
In Modeling Pt.2, two of the models (Random Forests and XGBoost) are themselves ensemble models:


*   Random Forests are an example of a **bagging** ensemble: they train many decision trees on bootstrapped samples and average their predictions to reduce variance.
*   XGBoost is an example of **boosting**: it builds an ensemble of trees sequentially, where each new tree focuses on correcting the errors of the previous ones.

We chose these models for three reasons:
1. Ensemble tree methods are known to perform well on structured data with many mixed-type features, which matches our UFO dataset after feature engineering.
2. They can naturally create non-linear decision boundaries, which is found useful after testing the baseline models.
3. They offer built-in mechanisms that deal with class imbalance.

---

### How we implemented it
We implemented and evaluated Random Forests and XGBoost under the same experimental framework as the other models:

Random Forests:
* We used `RandomForestClassifier` from scikit-learn.
* Key hyperparameters included:
  * `n_estimators` (number of trees),
  * `max_depth`, `min_samples_split`, `min_samples_leaf` (to control tree complexity and overfitting),
  * `class_weight="balanced"` for the original imbalanced training set.
* For the SMOTE scenario, we trained Random Forests on the balanced, oversampled training data and set `class_weight=None`, because the label distribution was already equalized by SMOTE.
* We evaluated both versions on the same original, imbalanced test set using accuracy and weighted macro F1.

XGBoost:
* We used `XGBClassifier` with a multi-class objective and set `num_class` equal to the number of UFO shapes.
* Important hyperparameters included:
  * `n_estimators`, `learning_rate`, `max_depth`,
  * `subsample`, `colsample_bytree`,
  * `tree_method="hist"` for efficiency.
* On the original training data, we incorporated class imbalance via **sample weights** computed from inverse class frequencies.
* On the SMOTE-resampled training data, we trained XGBoost without additional sample weights, since the data were already balanced.
* We evaluated both versions on the same original, imbalanced test set using accuracy and weighted macro F1.


### Results & Interpretation
Both Random Forests and XGBoost demonstrate typical ensemble behavior: they can fit the training data very well, but their generalization is not as satisfying as the training result, and depends heavily on the data distribution.

Random Forests:
* On the original training data with class weights, the tuned Random Forest achieved around 83% training accuracy and 0.835 weighted macro F1.
* On the test set, accuracy dropped to about 12.9% with 0.127 weighted macro F1, which is below the 21% majority-class baseline in accuracy, but better than the weighted macro F1 baseline.
* With SMOTE, Random Forests reached even higher training performance (about 94.7% accuracy and 0.949 weighted macro F1), but test accuracy stayed around 12.7% and 0.125 weighted macro F1.
* This pattern shows classic **overfitting** in an ensemble model: the forest can classify the balanced training data, yet it does not generalize well back to the original imbalanced distribution.

XGBoost:
* On the original training data with sample weights, XGBoost achieves about 24.5% training accuracy and 0.231 weighted macro F1, with only 7.3% test accuracy and 0.076 weighted macro F1.
* When trained on the SMOTE-resampled data, XGBoost benefited more clearly from the balanced distribution:
  * Training accuracy increased to about 38.9% with 0.367 weighted macro F1.
  * Test accuracy improved to about 16.4%, and 0.125 weighted macro F1.
* Although this is still below the 21% majority baseline in terms of accuracy, XGBoost with SMOTE is the better-performing model in our study and shows how a gradient-boosting ensemble model can perform in this scenario.

In summary, Random Forests and XGBoost serve as our main **ensemble models** in this project. The result shows both the strengths and limitations of ensemble learning on imbalanced, noisy, multi-class datasets.